In [1]:
!pip install -q langgraph langchain-groq langchain-core \
             groq chromadb sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [3]:
import os
from groq import Groq
from sentence_transformers import SentenceTransformer
import chromadb
from pypdf import PdfReader
import glob
from typing import TypedDict, Literal

# ── CONFIG ───────────────────────────────────────────────
GROQ_API_KEY = "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"
MODEL = "llama-3.1-8b-instant"

# ── LOAD PDFs ────────────────────────────────────────────
# Upload your 3 PDFs using the Colab file upload button
# (folder icon in left sidebar → upload)
pdf_files = glob.glob("*.pdf")

if not pdf_files:
    print("⚠ No PDFs found - upload your 3 pension PDFs using the folder icon")
else:
    print(f"Found: {pdf_files}")

    all_chunks = []
    for pdf_path in pdf_files:
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and len(text.strip()) > 50:
                words = text.split()
                chunk_size = 120
                for i in range(0, len(words), chunk_size - 15):
                    chunk = " ".join(words[i:i + chunk_size])
                    if len(chunk) > 100:
                        all_chunks.append({
                            "text": chunk,
                            "source": pdf_path,
                            "page": page_num + 1
                        })

    print(f"Created {len(all_chunks)} chunks")

    # ── EMBEDDINGS ───────────────────────────────────────
    print("Loading embedding model...")
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    chroma_client = chromadb.Client()
    try:
        chroma_client.delete_collection("pension_docs")
    except:
        pass

    collection = chroma_client.create_collection("pension_docs")

    batch_size = 50
    for i in range(0, len(all_chunks), batch_size):
        batch = all_chunks[i:i + batch_size]
        embeddings = embedder.encode([c["text"] for c in batch]).tolist()
        collection.add(
            documents=[c["text"] for c in batch],
            embeddings=embeddings,
            metadatas=[{"source": c["source"], "page": c["page"]} for c in batch],
            ids=[f"chunk_{i+j}" for j, _ in enumerate(batch)]
        )

    print(f"Stored {collection.count()} chunks in ChromaDB")

    # ── GROQ CLIENT ──────────────────────────────────────
    client = Groq(api_key=GROQ_API_KEY)
    test = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Say OK"}],
        max_tokens=5
    )
    print(f"LLM connected: {test.choices[0].message.content}")
    print("\n✓ All components ready. Proceed to Cell 3.")

Found: ['drawdown_pension_review.pdf', 'dc_pension_schemes.pdf', 'employer_pension_scheme.pdf']
Created 28 chunks
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stored 28 chunks in ChromaDB
LLM connected: OK.

✓ All components ready. Proceed to Cell 3.


In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

# ── AGENT STATE ──────────────────────────────────────────────────────────────
# TypedDict defines the data structure passed between agent nodes
# Every node reads from and writes to this shared state object
# Design decision: explicit state typing prevents runtime errors
# and makes the agent's behaviour auditable - important for governance

class AgentState(TypedDict):
    question: str           # original user question
    category: str           # classified query type
    context: str            # retrieved chunks from ChromaDB
    answer: str             # final answer
    needs_human: bool       # human-in-the-loop flag
    confidence: str         # HIGH / MEDIUM / LOW
    sources: list           # source documents used

# ── NODE 1: CLASSIFY ─────────────────────────────────────────────────────────
# First node — decides how to handle the query before retrieving anything
# Design decision: classify before retrieve to avoid wasting retrieval
# on queries the system cannot or should not handle autonomously

def classify_query(state: AgentState) -> AgentState:
    """
    Classifies the query into one of three categories:
    - ANSWERABLE: pension question likely in knowledge base
    - COMPLEX: multi-part or advice-seeking question needing human review
    - OUT_OF_SCOPE: nothing to do with pensions

    This is the routing logic of the agent — determines which path
    the query takes through the rest of the graph.
    """
    prompt = f"""Classify this question into exactly one category:
- ANSWERABLE: straightforward pension information question
- COMPLEX: requires regulated financial advice or is multi-part
- OUT_OF_SCOPE: not related to pensions at all

Question: {state['question']}

Respond with ONLY one word: ANSWERABLE, COMPLEX, or OUT_OF_SCOPE"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10
    )

    category = response.choices[0].message.content.strip().upper()

    # Validate — default to COMPLEX if unexpected response
    if category not in ["ANSWERABLE", "COMPLEX", "OUT_OF_SCOPE"]:
        category = "COMPLEX"

    return {**state, "category": category}


# ── NODE 2: RETRIEVE ─────────────────────────────────────────────────────────
def retrieve_context(state: AgentState) -> AgentState:
    """
    Retrieves relevant chunks from ChromaDB using semantic search.
    Only called for ANSWERABLE queries — not for COMPLEX or OUT_OF_SCOPE.

    Design decision: k=4 chunks balances context richness vs prompt length.
    Tested in Notebook 1 — this value gave best accuracy on our document set.
    """
    query_embedding = embedder.encode([state["question"]]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=4
    )

    context = "\n\n".join(results["documents"][0])
    sources = [f"{m['source']} p.{m['page']}"
               for m in results["metadatas"][0]]

    return {**state, "context": context, "sources": sources}


# ── NODE 3: GENERATE ANSWER ───────────────────────────────────────────────────
def generate_answer(state: AgentState) -> AgentState:
    """
    Generates a grounded answer using retrieved context.
    Also self-assesses confidence — critical for deciding whether
    to escalate to human review.

    Design decision: explicit confidence assessment built into the
    generation step rather than added as a separate call. Saves latency
    and keeps the governance signal close to the answer itself.
    """
    prompt = f"""You are a UK pension guidance assistant working for a regulated financial services firm.

Answer the question using ONLY the context below.
If the answer is not clearly in the context, say:
"I don't have enough information in my knowledge base to answer this accurately."

After your answer, on a new line write:
CONFIDENCE: HIGH, MEDIUM, or LOW
- HIGH: context directly and fully answers the question
- MEDIUM: context partially answers the question
- LOW: context is insufficient or tangentially related

Context:
{state['context']}

Question: {state['question']}

Answer:"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=400
    )

    full_response = response.choices[0].message.content

    # Parse answer and confidence score
    if "CONFIDENCE:" in full_response:
        parts = full_response.split("CONFIDENCE:")
        answer = parts[0].strip()
        confidence_line = parts[1].strip().upper()
        if "HIGH" in confidence_line:
            confidence = "HIGH"
        elif "MEDIUM" in confidence_line:
            confidence = "MEDIUM"
        else:
            confidence = "LOW"
    else:
        answer = full_response
        confidence = "LOW"  # Default to LOW if not provided

    # Flag for human review if confidence is LOW
    needs_human = confidence == "LOW"

    return {**state, "answer": answer, "confidence": confidence,
            "needs_human": needs_human}


# ── NODE 4: HUMAN ESCALATION ──────────────────────────────────────────────────
def escalate_to_human(state: AgentState) -> AgentState:
    """
    Human-in-the-loop node — triggered when:
    1. Query is classified as COMPLEX (requires regulated advice)
    2. Answer confidence is LOW (insufficient context)

    Design decision: explicit escalation is non-negotiable in a
    regulated pension context. An uncertain answer delivered with
    false confidence causes real customer harm.

    In production: this node would create a case in the CRM system,
    route to a qualified pension adviser, and log the escalation
    for compliance reporting.
    """
    if state["category"] == "COMPLEX":
        reason = "This query requires regulated financial advice"
    elif state["category"] == "OUT_OF_SCOPE":
        reason = "This query is outside the pension guidance scope"
    else:
        reason = f"Answer confidence is {state['confidence']} — insufficient context"

    escalation_message = f"""⚠ ESCALATED TO HUMAN ADVISER

Reason: {reason}

Query: {state['question']}

Next step: A qualified pension adviser will review this query.
Reference this conversation ID for continuity.

[In production: CRM ticket created, adviser notified, SLA timer started]"""

    return {**state, "answer": escalation_message, "needs_human": True}


# ── ROUTING LOGIC ─────────────────────────────────────────────────────────────
def route_after_classify(state: AgentState) -> Literal["retrieve", "escalate"]:
    """
    Routing function — determines next node after classification.
    Only ANSWERABLE queries proceed to retrieval.
    Everything else goes straight to escalation.
    """
    if state["category"] == "ANSWERABLE":
        return "retrieve"
    else:
        return "escalate"


def route_after_generate(state: AgentState) -> Literal["escalate", "__end__"]:
    """
    Routing function — determines next node after answer generation.
    LOW confidence answers are escalated rather than returned to user.
    HIGH and MEDIUM confidence answers are returned directly.
    """
    if state["needs_human"]:
        return "escalate"
    else:
        return "__end__"


# ── BUILD THE GRAPH ───────────────────────────────────────────────────────────
# LangGraph uses a directed graph structure
# Nodes = processing steps
# Edges = transitions between steps (can be conditional)
# Design decision: explicit graph structure makes agent behaviour
# auditable and testable — you can trace exactly which path
# any query took through the system

workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("classify", classify_query)
workflow.add_node("retrieve", retrieve_context)
workflow.add_node("generate", generate_answer)
workflow.add_node("escalate", escalate_to_human)

# Set entry point
workflow.set_entry_point("classify")

# Add conditional edges
workflow.add_conditional_edges(
    "classify",
    route_after_classify,
    {"retrieve": "retrieve", "escalate": "escalate"}
)

workflow.add_edge("retrieve", "generate")

workflow.add_conditional_edges(
    "generate",
    route_after_generate,
    {"escalate": "escalate", "__end__": END}
)

workflow.add_edge("escalate", END)

# Compile the graph
agent = workflow.compile()

print("✓ LangGraph agent compiled successfully")
print("\nAgent graph structure:")
print("  START → classify → [ANSWERABLE] → retrieve → generate → [HIGH/MED] → END")
print("                   → [COMPLEX]    → escalate → END")
print("                                  → [LOW confidence] → escalate → END")

✓ LangGraph agent compiled successfully

Agent graph structure:
  START → classify → [ANSWERABLE] → retrieve → generate → [HIGH/MED] → END
                   → [COMPLEX]    → escalate → END
                                  → [LOW confidence] → escalate → END


In [5]:
# ── RUN THE AGENT ─────────────────────────────────────────────────────────────
# Testing all three paths through the agent graph:
# Path 1: ANSWERABLE → retrieve → generate → HIGH/MEDIUM → END
# Path 2: COMPLEX → escalate → END
# Path 3: ANSWERABLE → retrieve → generate → LOW → escalate → END

test_queries = [
    # Path 1 expected: ANSWERABLE, high confidence
    "What is pension drawdown?",

    # Path 1 expected: ANSWERABLE, should answer from knowledge base
    "Can I take 25% of my pension tax free?",

    # Path 2 expected: COMPLEX, escalate immediately
    "Should I take drawdown or an annuity given my health condition?",

    # Path 2 expected: COMPLEX, requires regulated advice
    "I am 58 and have £200,000 in my pension. What should I do?",

    # Path 3 expected: ANSWERABLE but LOW confidence — knowledge gap
    "What happens to my defined benefit pension if my employer goes bust?",

    # Out of scope
    "What is the best mortgage deal available right now?"
]

print("=" * 65)
print("UK PENSION GUIDANCE AGENT — FULL TEST RUN")
print("Testing all agent paths including human escalation")
print("=" * 65)

for i, query in enumerate(test_queries, 1):
    print(f"\n{'=' * 65}")
    print(f"Query {i}: {query}")
    print(f"{'─' * 65}")

    # Initialise state
    initial_state = AgentState(
        question=query,
        category="",
        context="",
        answer="",
        needs_human=False,
        confidence="",
        sources=[]
    )

    # Run the agent
    result = agent.invoke(initial_state)

    # Display results
    print(f"Category:   {result['category']}")
    print(f"Confidence: {result['confidence'] if result['confidence'] else 'N/A'}")
    print(f"Escalated:  {'Yes' if result['needs_human'] else 'No'}")
    if result['sources']:
        print(f"Sources:    {result['sources'][:2]}")
    print(f"\nResponse:\n{result['answer']}")

print(f"\n{'=' * 65}")
print("END OF AGENT TEST RUN")
print("=" * 65)

UK PENSION GUIDANCE AGENT — FULL TEST RUN
Testing all agent paths including human escalation

Query 1: What is pension drawdown?
─────────────────────────────────────────────────────────────────
Category:   ANSWERABLE
Confidence: HIGH
Escalated:  No
Sources:    ['drawdown_pension_review.pdf p.3', 'drawdown_pension_review.pdf p.1']

Response:
Drawdown is also known as drawdown pension and income drawdown but is referred to as drawdown throughout this publication.

Query 2: Can I take 25% of my pension tax free?
─────────────────────────────────────────────────────────────────
Category:   ANSWERABLE
Confidence: HIGH
Escalated:  No
Sources:    ['drawdown_pension_review.pdf p.3', 'drawdown_pension_review.pdf p.3']

Response:
Yes, you can take 25% of your pension tax-free.

Query 3: Should I take drawdown or an annuity given my health condition?
─────────────────────────────────────────────────────────────────
Category:   COMPLEX
Confidence: N/A
Escalated:  Yes

Response:
⚠ ESCALATED TO HUM